# Gemma 4 31B V9.3: Zero-Tern5 / Zero-Cliff + True 60-Layer E2E Logit-KL QAT & Matched EAGLE-3 Assistant

- **Zero Ternary-5 Anywhere:** Replaces all 58 Attention layers (`L0..L57`) and upper-stack MLPs (`L36..L57`, plus `L0..L1`) with fast `2b_gauss COLBLOCK` (`0xA2`) NEON lookup kernels (`~6.72 GiB` total, `~6.50 GiB` text-resident).
- **True 60-Layer End-to-End Logit-KL + Cosine QAT:** Jointly optimizes all block scales `sc`, biases `bi`, STE INT8 low-rank residuals, RMSNorms, `layer_scalar`, and `final_norm` against exact Teacher Top-128 Softmax probabilities.
- **Matched `HeadRouter` (`20/32`) + `CRTR-V9` (`42/84`) + `C8DB` + `Assistant-V9` (`Q2`/`Q4` + `BHDR-384`).**

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/DannyB-bit/Language-U-Gemma4.git"
REPO_DIR = Path("/content/Language-U-Gemma4")
DRIVE_V93 = Path("/content/drive/MyDrive/gemma4_v9_3_e2e_qat")
DRIVE_V93.mkdir(parents=True, exist_ok=True)

if not REPO_DIR.is_dir():
    subprocess.run(["git", "clone", "-b", "v5-qat-dev", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", "v5-qat-dev"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/v5-qat-dev"], check=True)

print(subprocess.check_output(["git", "-C", str(REPO_DIR), "log", "-1", "--oneline"], text=True).strip(), flush=True)
print(subprocess.check_output(["nvidia-smi"], text=True), flush=True)

import torch
assert torch.cuda.is_available(), "CUDA A100/L4 GPU required!"
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"[GPU Ready] {torch.cuda.get_device_name(0)} | VRAM={vram_gb:.2f} GiB", flush=True)

_need_pip = False
try:
    import datasets, safetensors, tokenizers, huggingface_hub, PIL
    from transformers.models.gemma4.modeling_gemma4 import Gemma4VisionModel
except Exception:
    _need_pip = True
if _need_pip:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-U",
         "transformers>=5.0.0", "accelerate", "safetensors", "huggingface_hub", "datasets", "tokenizers", "pillow"],
        check=True,
    )

try:
    from google.colab import userdata
    hf_tok = userdata.get("HF_TOKEN")
    if hf_tok:
        os.environ["HF_TOKEN"] = hf_tok
except Exception:
    pass

SCRIPT = REPO_DIR / "experiments/run270_rust/calib/31b/run_v9_3_e2e_kl_qat_gpu.py"
res = subprocess.run([sys.executable, str(SCRIPT), "--selftest"], text=True, capture_output=True)
print(res.stdout, res.stderr, flush=True)
res.check_returncode()

In [ ]:
import os, subprocess, sys, time
from pathlib import Path

REPO_DIR = Path("/content/Language-U-Gemma4")
SCRIPT = REPO_DIR / "experiments/run270_rust/calib/31b/run_v9_3_e2e_kl_qat_gpu.py"
LOG_PATH = Path("/content/v9_3_train.log")
DRIVE_V93 = Path("/content/drive/MyDrive/gemma4_v9_3_e2e_qat")

env = os.environ.copy()
env["V9_3_DRIVE_DIR"] = str(DRIVE_V93)
env["V9_3_OUT_DIR"] = "/content/v9_3_outputs"
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:512"
env["COLAB_AUTO_UNASSIGN"] = "0"
env["PYTHONUNBUFFERED"] = "1"

with open(LOG_PATH, "w", encoding="utf-8") as lfp:
    proc = subprocess.Popen([sys.executable, "-u", str(SCRIPT), "--run", "--e2e-steps", "240"], stdout=lfp, stderr=subprocess.STDOUT, env=env)

seen = 0
while proc.poll() is None:
    time.sleep(15)
    if LOG_PATH.is_file():
        txt = LOG_PATH.read_text(encoding="utf-8", errors="replace")
        if len(txt) > seen:
            print(txt[seen:], end="", flush=True)
            seen = len(txt)
txt = LOG_PATH.read_text(encoding="utf-8", errors="replace")
if len(txt) > seen:
    print(txt[seen:], end="", flush=True)
assert proc.returncode == 0, f"V9.3 exited with {proc.returncode}"
print(subprocess.check_output(["sha256sum", "-c", "SHA256SUMS_V9_3.txt"], cwd=str(DRIVE_V93), text=True), flush=True)
print((DRIVE_V93 / "v9_3_report.json").read_text(encoding="utf-8"))